# Regression and general predictive learning

Family 6 fits reusable conditional mean, probability and quantile functions. This generated-data problem book demonstrates objective identities, held-out predictions and composition with family 4's transforms.

[Catalog and contracts](../docs/predictive_learning.md) · [Executable script](../examples/predictive_learning.py)

Install the project with `pip install -e '.[notebooks,neural]'` to run every cell. The neural section can be disabled with `RUN_NEURAL=False`; all other sections use the base dependencies. Training and test rows are explicitly chosen here.

In [ ]:
from pathlib import Path
import sys
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'convexity' / 'math').is_dir())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from numpy.testing import assert_allclose
from scipy.special import expit
from convexity.math import learning as learn, transforms as tx

RUN_NEURAL = True
rng = np.random.default_rng(42)
plt.rcParams.update({'figure.figsize': (9, 3.5), 'axes.grid': True, 'grid.alpha': .2})

## 1. Linear means and a shared regularization objective

Rows of `X` are observations, columns are features. Regression targets can be scalar or vector. By default, models fit an unpenalized intercept. Ridge, lasso and elastic net share the objective

$$\frac{1}{2n}\sum_i(y_i-b-x_i^T\beta)^2+\alpha\left[r\|\beta\|_1+\frac{1-r}{2}\|\beta\|_2^2\right].$$

For an orthogonal, centered design with $X^T X/n=I$, elastic net soft-thresholds the unpenalized coefficients and then shrinks them by $1+\alpha(1-r)$.

In [ ]:
orthogonal = np.vstack([np.eye(3), -np.eye(3)]) * np.sqrt(3)
beta = np.array([2., -.1, -3.])
alpha, ratio = .5, .4
model = learn.ElasticNet(alpha=alpha, l1_ratio=ratio).fit(orthogonal, 7 + orthogonal @ beta)
expected = np.sign(beta) * np.maximum(np.abs(beta) - alpha * ratio, 0) / (1 + alpha * (1-ratio))
assert_allclose(model.coefficients, expected)
assert_allclose(model.intercept, 7)
pd.DataFrame({'unpenalized': beta, 'elastic net': model.coefficients, 'analytic': expected})

In [ ]:
X = rng.normal(size=(1200, 3)) * [1., 3., .5]
beta = np.array([1.5, -2., .5])
y = 3 + X @ beta + .3 * rng.normal(size=len(X))
training, test = X[:900], X[900:]
specifications = {'OLS': learn.LinearRegression(), 'Ridge': learn.Ridge(alpha=.01),
                  'Lasso': learn.Lasso(alpha=.02), 'Elastic net': learn.ElasticNet(alpha=.02)}
models = {name: learn.Pipeline((tx.Standardize(),), tool).fit(training, y[:900])
          for name, tool in specifications.items()}
errors = {name: np.mean((model.predict(test)-y[900:])**2) for name, model in models.items()}
assert max(errors.values()) < .2
# A later fit must not alter an already fitted preprocessing/model snapshot.
before = models['Ridge'].predict(test)
learn.Pipeline((tx.Standardize(),), specifications['Ridge']).fit(training + 100, y[:900])
assert_allclose(models['Ridge'].predict(test), before)
pd.Series(errors, name='held-out MSE')

## 2. Generalized linear means and exposures

Poisson/log regression predicts a count mean. Exposure enters as an explicit log offset with coefficient one. A model trained with an offset requires an offset for every new prediction. Gaussian/identity and Gamma/log are also supported; binomial/logit is shown next.

In [ ]:
exposure = rng.uniform(.5, 2., len(X))
true_mean = np.exp(.5 + X @ [.2, -.08, .1]) * exposure
counts = rng.poisson(true_mean)
poisson = learn.GeneralizedLinearModel('poisson').fit(X, counts, offset=np.log(exposure))
predicted = poisson.predict(X, offset=np.log(exposure))
assert_allclose(predicted, np.exp(poisson.intercept + X @ poisson.coefficients) * exposure)
assert_allclose(poisson.coefficients, [.2, -.08, .1], atol=.1)
print(poisson.diagnostics)
pd.DataFrame({'true coefficients': [.2, -.08, .1], 'fitted coefficients': poisson.coefficients})

## 3. Binomial counts and logistic class probabilities

Grouped binomial regression takes successes and trial counts. Its conditional mean is a probability **per trial**; multiply by new trial counts for expected successes. `predict_proba` returns columns `[1-p,p]`, not probabilities over aggregate counts.

Logistic classification accepts arbitrary integer or string labels. Its probability columns follow `classes`. `predict` returns labels, whereas a binomial GLM's `predict` returns its conditional mean probability. Logistic `C` uses scikit-learn's inverse regularization convention.

In [ ]:
true_probability = expit(-.3 + X @ [.8, -.2, .4])
trials = rng.integers(10, 30, len(X))
successes = rng.binomial(trials, true_probability)
binomial = learn.GeneralizedLinearModel('binomial').fit(X, successes, trials=trials)
assert_allclose(binomial.predict_proba(test).sum(axis=-1), 1)
assert_allclose(binomial.coefficients, [.8, -.2, .4], atol=.09)
labels = np.where(rng.random(len(X)) < true_probability, 'success', 'failure')
logistic = learn.LogisticRegression().fit(training, labels[:900])
assert np.array_equal(logistic.classes, ['failure', 'success'])
probability = logistic.predict_proba(test)[:, 1]
assert np.array_equal(logistic.predict(test), logistic.classes[logistic.predict_proba(test).argmax(axis=-1)])
print('Probability columns:', logistic.classes)
print('Held-out mean squared probability error:', np.mean((probability-true_probability[900:])**2))

In [ ]:
fig, ax = plt.subplots()
ax.scatter(true_probability[900:], probability, alpha=.4, s=15, label='Logistic estimates')
ax.plot([0, 1], [0, 1], color='black', linestyle='--', label='Generating probabilities')
ax.set(xlabel='Generating probability', ylabel='Fitted probability', title='Held-out conditional probabilities')
ax.legend()
plt.show()

## 4. Conditional quantiles

We generate heteroskedastic uniform noise so the true conditional quantiles are linear and known. Each requested quantile minimizes pinball loss independently. A median is not relabeled a mean, and independent curves may cross outside this simple example. No sorting or interpolation is applied.

In [ ]:
qx = rng.uniform(-1, 1, (2000, 1))
scale = 1 + .4 * qx[:, 0]
qy = 2 + 3 * qx[:, 0] + scale * rng.uniform(-1, 1, len(qx))
levels = np.array([.1, .5, .9])
quantile = learn.QuantileRegression(tuple(levels)).fit(qx[:1500], qy[:1500])
grid = np.linspace(-1, 1, 100)[:, None]
truth = 2 + 3 * grid + (1 + .4 * grid) * (2*levels-1)
estimated = quantile.predict_quantiles(grid)
assert np.max(np.abs(estimated-truth)) < .16
coverage = np.mean(qy[1500:, None] <= quantile.predict_quantiles(qx[1500:]), axis=0)
pd.DataFrame({'quantile': levels, 'held-out coverage': coverage})

In [ ]:
fig, ax = plt.subplots()
ax.scatter(qx[1500:, 0], qy[1500:], s=9, alpha=.15, color='gray', label='Held-out observations')
for j, level in enumerate(levels):
    line, = ax.plot(grid[:, 0], estimated[:, j], label=f'Fitted q={level:.1f}')
    ax.plot(grid[:, 0], truth[:, j], color=line.get_color(), linestyle='--', alpha=.7)
ax.set(xlabel='Feature', ylabel='Response', title='Conditional quantiles (dashed: generating curves)')
ax.legend(ncol=2)
plt.show()

## 5. Nonlinear means from trees and ensembles

The response is a quadratic feature plus a threshold jump. Tree, random forest, extra trees and gradient boosting fits use only the first 700 rows. This comparison is an executable recovery check, not a general ranking of learners. Seeds are explicit and fixed.

In [ ]:
tree_x = rng.uniform(-2, 2, (900, 2))
tree_y = tree_x[:, 0]**2 + (tree_x[:, 1] > 0)
tools = [learn.DecisionTree(max_depth=6), learn.RandomForest(n_estimators=40),
         learn.ExtraTrees(n_estimators=40), learn.GradientBoosting(n_estimators=100)]
tree_models = {type(tool).__name__: tool.fit(tree_x[:700], tree_y[:700]) for tool in tools}
mse = {name: np.mean((model.predict(tree_x[700:])-tree_y[700:])**2) for name, model in tree_models.items()}
assert max(mse.values()) < .12
forest_classifier = learn.RandomForest(task='classification', n_estimators=30).fit(training, labels[:900])
assert_allclose(forest_classifier.predict_proba(test).sum(axis=-1), 1)
pd.Series(mse, name='held-out MSE')

## 6. Optional neural functions

PyTorch supplies automatic differentiation and Adam. This CPU float64 learner uses full-batch training and an explicit gradient tolerance; reaching the iteration budget does not count as convergence. `loss_history` includes both initial and final loss. A supplied `torch.nn.Module` can also be snapshotted with `neural_function`, using `(n,m)` outputs or class logits.

Set `RUN_NEURAL=False` above to run the notebook without PyTorch. The thread setting below is local to this worked example and restored after fitting.

In [ ]:
if RUN_NEURAL:
    import torch
    previous_threads = torch.get_num_threads()
    try:
        torch.set_num_threads(1)
        neural_x = np.linspace(-2, 2, 120)[:, None]
        neural_y = np.sin(neural_x[:, 0])
        network = learn.NeuralRegressor(hidden_layers=(12,), seed=7, maxiter=3000, tolerance=3e-4).fit(neural_x, neural_y)
        neural_prediction = network.predict(neural_x)
        assert np.mean((neural_prediction-neural_y)**2) < .001
        print(network.diagnostics)
        fig, axes = plt.subplots(1, 2, figsize=(10, 3.3), layout='constrained')
        axes[0].plot(neural_x[:, 0], neural_y, label='Generating function')
        axes[0].plot(neural_x[:, 0], neural_prediction, '--', label='Neural prediction')
        axes[0].set(xlabel='Feature', ylabel='Response', title='Dense neural conditional mean')
        axes[0].legend()
        axes[1].semilogy(network.loss_history)
        axes[1].set(xlabel='Adam updates', ylabel='Training loss', title='Objective history')
        plt.show()
    finally:
        torch.set_num_threads(previous_threads)
else:
    print('Optional neural section disabled.')

## 7. A user-written predictor

An adapter states its input dimension, output shape and supported mathematical operations. The facade checks shape and finiteness and preserves arbitrary leading prediction axes. No registry or subclass is required. Unsupported outputs raise an explicit error.

In [ ]:
class ConstantAdapter:
    input_dimension = 2
    output_shape = ()
    capabilities = frozenset({'mean'})
    def evaluate(self, operation, features, *, offset=None):
        return np.full(len(features), 3.)

class CustomLearner:
    def fit(self, features, response):
        return learn.FittedModel(ConstantAdapter())

custom = learn.fit([[1., 2.], [3., 4.]], [2., 4.], CustomLearner())
assert_allclose(custom.predict(np.zeros((2, 3, 2))), 3)
try:
    custom.predict_proba([[1., 2.]])
except NotImplementedError as error:
    print(error)
print('All family-6 notebook assertions passed.')